# ipadic-neologd

## 依存関係のインストール

In [ ]:
import os
from pathlib import Path


def find_project_root() -> Path:
    """pyproject.tomlがあるプロジェクトルートを返す。"""
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("pyproject.toml があるディレクトリで notebook を開いてください")


PROJECT_ROOT = find_project_root()
TOOLS_DIR = PROJECT_ROOT / ".cache" / "mecab"
MECAB_PREFIX = TOOLS_DIR / "local"
NEOLOGD_DIR = TOOLS_DIR / "dic" / "mecab-ipadic-neologd"
MECAB_URL = "https://deb.debian.org/debian/pool/main/m/mecab/mecab_0.996.orig.tar.gz"
NEOLOGD_URL = "https://api.github.com/repos/neologd/mecab-ipadic-neologd/tarball/v0.0.7"

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["TOOLS_DIR"] = str(TOOLS_DIR)
os.environ["MECAB_PREFIX"] = str(MECAB_PREFIX)
os.environ["NEOLOGD_DIR"] = str(NEOLOGD_DIR)
os.environ["MECAB_URL"] = MECAB_URL
os.environ["NEOLOGD_URL"] = NEOLOGD_URL

PROJECT_ROOT


In [ ]:
%%bash
set -euxo pipefail
unset CLICOLOR_FORCE CLICOLOR LSCOLORS LS_COLORS

MECAB_TARBALL="$TOOLS_DIR/source/mecab.tar.gz"

rm -rf "$TOOLS_DIR/source/mecab-0.996" "$MECAB_PREFIX" "$MECAB_TARBALL"
mkdir -p "$TOOLS_DIR/source"

curl -fL "$MECAB_URL" -o "$MECAB_TARBALL"
tar -xzf "$MECAB_TARBALL" -C "$TOOLS_DIR/source"

cd "$TOOLS_DIR/source/mecab-0.996"
./configure --prefix="$MECAB_PREFIX" --with-charset=utf8
make -j"$(python -c 'import os; print(os.cpu_count() or 2)')"
make install


In [ ]:
%%bash
set -euxo pipefail
unset CLICOLOR_FORCE CLICOLOR LSCOLORS LS_COLORS

NEOLOGD_TARBALL="$TOOLS_DIR/source/mecab-ipadic-neologd.tar.gz"

rm -rf "$TOOLS_DIR/source/mecab-ipadic-neologd" "$NEOLOGD_DIR" "$NEOLOGD_TARBALL"
mkdir -p "$TOOLS_DIR/source/mecab-ipadic-neologd" "$TOOLS_DIR/dic"

export PATH="$MECAB_PREFIX/bin:$PATH"
curl -fL "$NEOLOGD_URL" -o "$NEOLOGD_TARBALL"
tar -xzf "$NEOLOGD_TARBALL" -C "$TOOLS_DIR/source/mecab-ipadic-neologd" --strip-components 1

cd "$TOOLS_DIR/source/mecab-ipadic-neologd"
python - <<'PY'
from pathlib import Path

script = Path("libexec/make-mecab-ipadic-neologd.sh")
text = script.read_text()
old = 'cd ${NEOLOGD_DIC_DIR}\n\nMECAB_PATH=`which mecab`'
new = 'cd ${NEOLOGD_DIC_DIR}\nfind . -type f -exec touch -t 200001010000 {} +\n\nMECAB_PATH=`which mecab`'
if old not in text:
    raise RuntimeError("patch target was not found in make-mecab-ipadic-neologd.sh")
script.write_text(text.replace(old, new, 1))
PY
./bin/install-mecab-ipadic-neologd -n -y -p "$NEOLOGD_DIR"


## 実装コード

In [37]:
import MeCab
import unicodedata
from dataclasses import dataclass
from IPython.display import Markdown, display


@dataclass
class Morpheme:
    surface: str
    part_of_speech: str
    base_form: str
    reading: str
    mora_count: int


@dataclass
class SenryuResult:
    word: str
    is_senryu: bool
    total_moras: int
    cumulative_counts: list[int]
    morphemes: list[Morpheme]

    @property
    def surfaces(self) -> str:
        """形態素の表層形を表示用の文字列にまとめる。"""
        return " / ".join(morpheme.surface for morpheme in self.morphemes)

    @property
    def readings(self) -> str:
        """形態素の読みと音数を表示用の文字列にまとめる。"""
        return " / ".join(
            f"{morpheme.reading}({morpheme.mora_count})" for morpheme in self.morphemes
        )


FIRST_PHRASE_MORAS = 5
SECOND_PHRASE_MORAS = 7
THIRD_PHRASE_MORAS = 5


NON_MORA_SMALL_KANA = set("ァィゥェォャュョヮ")


def is_kana_or_long_vowel_mark(char: str) -> bool:
    """1文字が仮名または長音記号ならTrueを返す。"""
    return char == "ー" or "\u3041" <= char <= "\u3096" or "\u30a1" <= char <= "\u30fa"


def count_reading_moras(reading: str) -> int:
    """読みの文字列から音数を数える。"""
    normalized = unicodedata.normalize("NFKC", reading)
    return sum(
        1
        for char in normalized
        if is_kana_or_long_vowel_mark(char) and char not in NON_MORA_SMALL_KANA
    )


def has_mecab_column(mecab_columns: list[str], index: int) -> bool:
    """node.featureをカンマで分割した列が指定位置にあり、未定義でなければTrueを返す。"""
    return len(mecab_columns) > index and mecab_columns[index] != "*"


def resolve_reading(surface: str, mecab_columns: list[str]) -> str:
    """MeCabの読みを返し、読めない場合は表層形から仮名の読みを補う。"""
    if has_mecab_column(mecab_columns, 7):
        return mecab_columns[7]

    if any(is_kana_or_long_vowel_mark(char) for char in surface):
        return surface

    raise ValueError(f"読みを取得できません: {surface}")


def resolve_base_form(surface: str, mecab_columns: list[str]) -> str:
    """MeCabの原形を返し、なければ表層形を返す。"""
    if has_mecab_column(mecab_columns, 6):
        return mecab_columns[6]
    return surface


def has_senryu_mora_counts(cumulative_counts: list[int], total_moras: int) -> bool:
    """音数の累計が5-7-5の切れ目と合計に合うならTrueを返す。"""
    # 5音目と12音目で形態素の区切りが来て、全体が17音なら5-7-5とみなす。
    first_break_moras = FIRST_PHRASE_MORAS
    second_break_moras = FIRST_PHRASE_MORAS + SECOND_PHRASE_MORAS
    expected_total_moras = FIRST_PHRASE_MORAS + SECOND_PHRASE_MORAS + THIRD_PHRASE_MORAS

    return (
        first_break_moras in cumulative_counts
        and second_break_moras in cumulative_counts
        and total_moras == expected_total_moras
    )


def split_morphemes(word: str, tagger: MeCab.Tagger) -> list[Morpheme]:
    """入力文を形態素に分け、読みと音数を付けて返す。"""
    morphemes = []
    node = tagger.parseToNode(word)
    while node:
        if node.surface:
            mecab_columns = node.feature.split(",")
            reading = resolve_reading(node.surface, mecab_columns)
            morphemes.append(
                Morpheme(
                    surface=node.surface,
                    part_of_speech=mecab_columns[0] if len(mecab_columns) > 0 else "",
                    base_form=resolve_base_form(node.surface, mecab_columns),
                    reading=reading,
                    mora_count=count_reading_moras(reading),
                )
            )
        node = node.next

    return morphemes


def analyze_senryu(word: str, tagger: MeCab.Tagger) -> SenryuResult:
    """入力文を解析し、川柳かどうかと音数の情報を返す。"""
    morphemes = split_morphemes(word, tagger)
    cumulative_counts = []
    total_moras = 0
    for morpheme in morphemes:
        total_moras += morpheme.mora_count
        cumulative_counts.append(total_moras)

    is_senryu = has_senryu_mora_counts(cumulative_counts, total_moras)
    return SenryuResult(
        word=word,
        is_senryu=is_senryu,
        total_moras=total_moras,
        cumulative_counts=cumulative_counts,
        morphemes=morphemes,
    )


def format_senryu_results_table(test_cases: list[tuple[str, bool]], results: list[SenryuResult]) -> str:
    """テストケースと解析結果をMarkdownの表に整形する。"""
    headers = ["入力", "期待", "判定", "確認", "音数", "累計", "読み"]
    rows = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]

    for (_, expected), result in zip(test_cases, results):
        passed = result.is_senryu == expected
        rows.append(
            "| "
            + " | ".join(
                [
                    result.word,
                    "川柳" if expected else "違う",
                    "川柳" if result.is_senryu else "違う",
                    "OK" if passed else "NG",
                    str(result.total_moras),
                    ", ".join(str(count) for count in result.cumulative_counts),
                    result.readings,
                ]
            )
            + " |"
        )

    return "\n".join(rows)


def run_senryu_table(test_cases: list[tuple[str, bool]], tagger: MeCab.Tagger) -> list[SenryuResult]:
    """テストケースを解析し、表を表示して結果を返す。"""
    results = [analyze_senryu(word, tagger) for word, _ in test_cases]
    display(Markdown(format_senryu_results_table(test_cases, results)))
    return results


## 実行

In [38]:
if not NEOLOGD_DIR.is_dir():
    raise FileNotFoundError(f"辞書がありません。install-ipadic-neologd セルを実行してください: {NEOLOGD_DIR}")

tagger = MeCab.Tagger(f"-r /dev/null -d {NEOLOGD_DIR}")
tagger.parse("")

test_cases = [
    ("朝起きて今日も会社に遅刻する", True),
    ("推し活で財布の中が空になる", True),
    ("コンビニで財布忘れて立ち尽くす", True),
    ("推し活", False),
    ("朝起きて猫にご飯を催促される", False),
]

results = run_senryu_table(test_cases, tagger)
results


| 入力 | 期待 | 判定 | 確認 | 音数 | 累計 | 読み |
| --- | --- | --- | --- | --- | --- | --- |
| 朝起きて今日も会社に遅刻する | 川柳 | 川柳 | OK | 17 | 2, 4, 5, 7, 8, 11, 12, 15, 17 | アサ(2) / オキ(2) / テ(1) / キョウ(2) / モ(1) / カイシャ(3) / ニ(1) / チコク(3) / スル(2) |
| 推し活で財布の中が空になる | 川柳 | 川柳 | OK | 17 | 2, 4, 5, 8, 9, 11, 12, 14, 15, 17 | オシ(2) / カツ(2) / デ(1) / サイフ(3) / ノ(1) / ナカ(2) / ガ(1) / ソラ(2) / ニ(1) / ナル(2) |
| コンビニで財布忘れて立ち尽くす | 川柳 | 川柳 | OK | 17 | 4, 5, 8, 11, 12, 17 | コンビニ(4) / デ(1) / サイフ(3) / ワスレ(3) / テ(1) / タチツクス(5) |
| 推し活 | 違う | 違う | OK | 4 | 2, 4 | オシ(2) / カツ(2) |
| 朝起きて猫にご飯を催促される | 違う | 違う | OK | 19 | 2, 4, 5, 7, 8, 11, 12, 16, 17, 19 | アサ(2) / オキ(2) / テ(1) / ネコ(2) / ニ(1) / ゴハン(3) / ヲ(1) / サイソク(4) / サ(1) / レル(2) |

[SenryuResult(word='朝起きて今日も会社に遅刻する', is_senryu=True, total_moras=17, cumulative_counts=[2, 4, 5, 7, 8, 11, 12, 15, 17], morphemes=[Morpheme(surface='朝', part_of_speech='名詞', base_form='朝', reading='アサ', mora_count=2), Morpheme(surface='起き', part_of_speech='動詞', base_form='起きる', reading='オキ', mora_count=2), Morpheme(surface='て', part_of_speech='助詞', base_form='て', reading='テ', mora_count=1), Morpheme(surface='今日', part_of_speech='名詞', base_form='今日', reading='キョウ', mora_count=2), Morpheme(surface='も', part_of_speech='助詞', base_form='も', reading='モ', mora_count=1), Morpheme(surface='会社', part_of_speech='名詞', base_form='会社', reading='カイシャ', mora_count=3), Morpheme(surface='に', part_of_speech='助詞', base_form='に', reading='ニ', mora_count=1), Morpheme(surface='遅刻', part_of_speech='名詞', base_form='遅刻', reading='チコク', mora_count=3), Morpheme(surface='する', part_of_speech='動詞', base_form='する', reading='スル', mora_count=2)]),
 SenryuResult(word='推し活で財布の中が空になる', is_senryu=True, total_moras=17, cumu